# Chat with the LLMs

This notebook sets up a chat interface with a chosen model (base or finetuned) and provides the output with and without RAG.

## Prerequisites

The RAG implementation requires the vector database is pre-generated. The GitHub Actions workflow should keep it updated.

If not, run the [document generation script](retrieval_db_update.ipynb).

Using the fine-tuned model requires the chat model has been trained on the data.

If it hasn't been run yet, run the [model finetuning notebook](finetuning.ipynb).

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from peft import PeftModel
from retrieval import Retrieval

In [2]:
# Whether to apply the fine-tuned LoRA adapter to the weights.
LORA = True
# Whether to quantize the base model weights to reduce the memory footprint.
QUANTIZATION = True
# Number of documents to query from the vector store.
TOPK = 3

# Chat LLM model name
CHAT_MODEL = "Qwen/Qwen2.5-7B-Instruct"
# Fine-tuned LoRA adapter directory
LORA_ADAPTER = f"/opt/shared/lora/{CHAT_MODEL}-Finetuned"

In [3]:
# Check GPU availability
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA Available: True
GPU: NVIDIA A100 80GB PCIe


In [4]:
# Use default embeddings model, sentence-transformers/all-distilroberta-v1
# Use the default ChromaDB directory and collection
retrieval = Retrieval(log=True)

Loading the sentence transformer, sentence-transformers/all-distilroberta-v1 ...
Loading ChromaDB, /opt/shared/data/chromadb ...
Loading collection, all-documents ...
Initializing retrieval done


In [5]:
# Load the model and tokenizer

print(f"Loading model and tokenizer: {CHAT_MODEL}")

chat_tokenizer = AutoTokenizer.from_pretrained(
    CHAT_MODEL,
)
# Ensure padding token is set
if chat_tokenizer.pad_token is None:
    chat_tokenizer.pad_token = chat_tokenizer.eos_token

print("Tokenizer loaded")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    CHAT_MODEL,
    quantization_config=bnb_config if QUANTIZATION else None,
    dtype=torch.bfloat16 if not QUANTIZATION else None,
    trust_remote_code=True,
    device_map="auto"
)

print(base_model.get_memory_footprint() / 1024**3)  # Memory in GiB
print(f"VRAM after model load: {torch.cuda.memory_allocated() / 1024**3:.2f} GiB")
print(f"Reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GiB")

Loading model and tokenizer: Qwen/Qwen2.5-7B-Instruct
Tokenizer loaded


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

5.06946873664856
VRAM after model load: 5.49 GiB
Reserved: 6.97 GiB


In [7]:
if LORA:
    chat_model = PeftModel.from_pretrained(base_model, LORA_ADAPTER)
else:
    chat_model = base_model

print("Model loaded")

print(f"VRAM after model load: {torch.cuda.memory_allocated() / 1024**3:.2f} GiB")
print(f"Reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GiB")

# Create the pipeline
pipe = pipeline(
    "text-generation",
    model=chat_model,
    tokenizer=chat_tokenizer,
    device_map="auto"
)

Device set to use cuda:0


Model loaded
VRAM after model load: 5.64 GiB
Reserved: 7.30 GiB


In [8]:
import copy
# adds some nice features to `input`
import readline

# Maintain conversation history
conversation = [
    {"role": "system", "content": "You are a helpful assistant."},
]

# Maintain RAG conversation history
rag_conversation = copy.deepcopy(conversation)
rag_conversation[0]["content"] += "\n\nAdditional context documents may be included with queries. Integrate relevant information from these documents seamlessly into comprehensive responses. When documents are unhelpful or off-topic, rely on your existing knowledge. Maintain your normal level of detail and insight regardless of document quality."

# RAG document injection history
history = set()

while True:
    user_input = input("> ")

    if not user_input or user_input in ["quit", "q", "exit"]:
        break

    for rag, convo in [(False, conversation), (True, rag_conversation)]:
        if rag:
            user_input = retrieval.augment(user_input, TOPK, history, log=True)
        
        # Add user message to conversation
        convo.append({"role": "user", "content": user_input})

        # Pass the conversation directly to the pipeline
        outputs = pipe(
            convo,
            max_new_tokens=1024,
            # To ensure repeatability, always pick the most-likely candidate for the next token.
            # To achieve this, turn sampling off (~= setting temperature to zero, but torch doesn't like that)
            do_sample=False,
            temperature=None,
            top_p=None,
            top_k=None
        )
    
        # Extract the assistant's response
        response = outputs[0]["generated_text"][-1]["content"]
    
        # Add assistant response to conversation
        convo.append({"role": "assistant", "content": response})
        
        print()
        if rag:
            print("RAG+", end="")
        print("LLM>", response)
        print()


>  How many r's are there in the word strrawberry?



LLM> In the word "strawberry," there are three 'r's.

[RAG] IRRELEVANT (0.9596561789512634): NOT INJECTING FILE: tutorial2_readme-wirguard-vpn-cluster-access-1.md
[RAG] IRRELEVANT (0.9652928113937378): NOT INJECTING FILE: tutorial3_readme-linpack-theoretical-peak-performance-top500-list-1.md
[RAG] IRRELEVANT (0.9727678298950195): NOT INJECTING FILE: tutorial1_readme-launching-your-first-open-stack-virtual-machine-instance-linux-flavors-and-distributions-summary-of-linux-distributions-1.md

RAG+LLM> There are two 'r's in the word strawberry.



>  q


In [9]:
import gc

# Delete the GPU-hogging resources
del pipe
del chat_model
del base_model
del chat_tokenizer

# Force gc
gc.collect()

# Clear IPython's execution result cache
ip = get_ipython()
ip.displayhook.flush()

# Clear all In/Out history
ip.history_manager.reset(new_session=False)
ip.history_manager.input_hist_parsed[:] = []
ip.history_manager.input_hist_raw[:] = []
ip.history_manager.output_hist.clear()
ip.history_manager.output_hist_reprs.clear()
ip.history_manager.dir_hist[:] = []

# Clear the user namespace cache
ip.user_ns_hidden.clear()

# Force gc
gc.collect()

# More aggressive cache clearing
torch.cuda.empty_cache()
torch.cuda.synchronize()  # Wait for all operations to complete
torch.cuda.empty_cache()

In [10]:
# %reset

In [11]:
# It's more reliable to just reset the kernel than hoping the above work
# probably not good for CI/CD though.

# import IPython
# IPython.Application.instance().kernel.do_shutdown(restart=True)  